# Does chain-of-thought help? Measure it.

**Session 4 · Track A · local Ollama**

Compare CoT vs direct answering on your eval set — decide from numbers.

In [1]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from utils import ask
from eval import load_cases, run_eval, print_report, exact


### Worked example

Direct answering vs chain-of-thought on a small reasoning set, scored the same way, with the token cost of CoT measured.


In [2]:
# Worked example: CoT vs direct - decide from numbers
import re
from utils import count_tokens

CASES = [
    {"input": "A shirt costs $40 after a 20% discount. Original price?",        "expected": "50"},
    {"input": "If 3 pens cost $2.10, what do 10 pens cost in dollars?",         "expected": "7"},
    {"input": "Tom is twice as old as Sara. In 5 years their ages sum to 40. Tom's age now?", "expected": "20"},
]

DIRECT = "{q}\nAnswer with a number only."
COT    = "{q}\nThink step by step, then end with a line 'FINAL: <number>'."

def last_number(s):
    nums = re.findall(r"-?\d+(?:\.\d+)?", s)
    return nums[-1] if nums else s.strip()

def score(template, extract):
    ok = tokens = 0
    for c in CASES:
        out = ask(template.format(q=c["input"]))
        tokens += count_tokens(out)
        ok += last_number(extract(out)) == c["expected"]
    return ok, len(CASES), tokens

d_ok, n, d_tok = score(DIRECT, lambda s: s)
c_ok, _, c_tok = score(COT,    lambda s: s.split("FINAL:")[-1])
print(f"direct: {d_ok}/{n}  ~{d_tok} output tokens")
print(f"CoT:    {c_ok}/{n}  ~{c_tok} output tokens  ({c_tok - d_tok:+d} vs direct)")


direct: 3/3  ~3 output tokens
CoT:    3/3  ~497 output tokens  (+494 vs direct)


## Your turn - vary the example

1. Add 3 harder problems where you expect direct answering to fail.
2. Try a middle option: "give a one-line reason, then the answer". Where does it land?
3. Keep the winner. Is the extra CoT token cost worth the accuracy gain here?


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
